In [ ]:
import numpy as np
import nir

from spinnaker2 import hardware
from spinnaker2 import s2_nir


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = "can_ids_spinnaker2.nir"

# 100 timesteps = same as your notebook
TIMESTEPS = 100

# SpiNNaker timestep
DT = 0.001   # 1 ms


# ============================================================
# LOAD NIR MODEL
# ============================================================

print("=" * 60)
print("LOADING NIR MODEL")
print("=" * 60)

nir_model = nir.read(MODEL_PATH)

print(nir_model)


# ============================================================
# SPI NNAKER-2 CONVERSION CONFIGURATION
# ============================================================

config = s2_nir.ConversionConfig()

# Record output spikes and voltage
config.output_record = [
    "spikes",
    "v"
]

# 1 ms timestep
config.dt = DT

# No additional synaptic delay
config.conn_delay = 0

# Scale weights into SpiNNaker-2 hardware range
config.scale_weights = True

# Use all weights when determining scaling
config.weight_scale_percentile = 100

# Your NIR model uses IF neurons
# Reset voltage to zero after spike
config.reset = s2_nir.ResetMethod.ZERO

# Forward Euler integration
config.integrator = s2_nir.IntegratorMethod.FORWARD


# ============================================================
# CONVERT NIR -> SPINNAKER-2 NETWORK
# ============================================================

print("=" * 60)
print("CONVERTING NIR -> SPINNAKER-2")
print("=" * 60)

net, input_pops, output_pops = s2_nir.from_nir(
    nir_model,
    config
)

print("Conversion successful")

print("Number of input populations:",
      len(input_pops))

print("Number of output populations:",
      len(output_pops))


# ============================================================
# INPUT POPULATION
# ============================================================

input_pop = input_pops[0]

print("Input population:")
print(input_pop)


# ============================================================
# CREATE TEST INPUT
# ============================================================

# Your model input is:
#
# 1 × 54 × 8
#
# therefore:
#
# 1 * 54 * 8 = 432 neurons

N_INPUT = 1 * 54 * 8

print("Input neurons:", N_INPUT)


# ------------------------------------------------------------
# Example spike pattern
#
# Replace this with your real CAN sample.
# ------------------------------------------------------------

spike_times = {
    neuron: []
    for neuron in range(N_INPUT)
}


# Example:
# stimulate neuron 0 at timestep 10
# stimulate neuron 20 at timestep 20

spike_times[0] = [10]
spike_times[20] = [20]


# Give spike times to the input population
input_pop.params = spike_times


# ============================================================
# HARDWARE
# ============================================================

print("=" * 60)
print("STARTING SPINNAKER-2")
print("=" * 60)

hw = hardware.SpiNNaker2Chip()


# ============================================================
# RUN
# ============================================================

hw.run(
    net,
    TIMESTEPS
)


# ============================================================
# OUTPUT
# ============================================================

print("=" * 60)
print("READING OUTPUT")
print("=" * 60)

output_pop = output_pops[0]

spikes = output_pop.get_spikes()

voltages = output_pop.get_voltages()


# ============================================================
# PRINT OUTPUT SPIKES
# ============================================================

print("\nOutput spikes:")

for neuron, times in spikes.items():

    print(
        f"Neuron {neuron}: {times}"
    )


# ============================================================
# CLASSIFICATION
# ============================================================

spike_count = np.zeros(2)

for neuron, times in spikes.items():

    if neuron < 2:

        spike_count[neuron] = len(times)


prediction = int(
    np.argmax(spike_count)
)


print("\n" + "=" * 60)
print("CLASSIFICATION")
print("=" * 60)

print("Class 0 = R")
print("Class 1 = T")

print("Spike counts:", spike_count)

print("Prediction:", prediction)

if prediction == 0:
    print("Prediction: R (Normal)")
else:
    print("Prediction: T (Attack)")